# 🎵 Riffusion Kaggle Song Studio

## Notebook-as-a-Service: Remote-Controlled Long-Form AI Music Generation

This notebook sets up a FastAPI server that exposes the Riffusion song generation pipeline,
accessible via a Cloudflare tunnel from your desktop/mobile app.

### Features:
- **5-10 Minute Songs**: Parses lyrics into structured sections (Intro, Verse, Chorus, Bridge, Outro)
- **Seamless Stitching**: Crossfades and normalizes audio for professional results
- **Checkpointing**: Resume generation if notebook disconnects
- **REST API**: Full remote control via HTTP endpoints
- **No Cost**: Uses free Kaggle T4 GPU resources

### Quick Start:
1. Run all cells in order
2. Copy the public tunnel URL
3. Use the client SDK or REST API to generate songs

## Step 1: Install Dependencies

In [ ]:
# Install system dependencies
!apt-get update -qq
!apt-get install -y -qq ffmpeg

# Install Python packages
!pip install -q \
    torch torchaudio \
    transformers diffusers accelerate \
    pydub librosa soundfile \
    fastapi uvicorn python-multipart \
    pyyaml requests pillow numpy scipy

print("✅ Dependencies installed")

## Step 2: Install Cloudflared for Tunnel

In [ ]:
# Download and install cloudflared
!curl -L --output /tmp/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i /tmp/cloudflared.deb

print("✅ Cloudflared installed")

## Step 3: Setup Source Package

In [ ]:
import sys
import os

# Add the src directory to Python path
NOTEBOOK_DIR = '/kaggle/working/kaggle_riffusion/notebook'
SRC_DIR = os.path.join(NOTEBOOK_DIR, 'src')

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Verify imports work
from preset_engine import PresetEngine
from song_arranger import SongArranger
from long_form_generator import LongFormGenerator
from audio_stitcher import stitch_song_sections
from api_server import app
from tunnel_manager import TunnelManager

print("✅ Source package configured")
print(f"   Available presets: {PresetEngine().list_presets()}")

## Step 4: Check GPU Environment

In [ ]:
import torch

print("=== GPU Environment ===")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("⚠️ No GPU detected - generation will be very slow")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {device}")

## Step 5: Start FastAPI Server with Tunnel

In [ ]:
import threading
import time
import uvicorn
from tunnel_manager import TunnelManager

# Configuration
PORT = 8000
OUTPUT_DIR = "/kaggle/working/riffusion_output"

# Initialize tunnel manager
tunnel_mgr = TunnelManager()

# Start the API server in a background thread
def run_server():
    uvicorn.run(
        "api_server:app",
        host="0.0.0.0",
        port=PORT,
        log_level="info"
    )

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Wait for server to start
time.sleep(2)

# Start cloudflared tunnel
print("Starting Cloudflare tunnel...")
success, tunnel_url = tunnel_mgr.start_tunnel(PORT)

if success and tunnel_url:
    print("\n" + "="*60)
    print("🎉 SERVER READY!")
    print("="*60)
    print(f"\n📡 Public API URL: {tunnel_url}")
    print(f"\n💡 Usage:")
    print(f"   GET  {tunnel_url}/health - Health check")
    print(f"   GET  {tunnel_url}/presets - List presets")
    print(f"   POST {tunnel_url}/generate_song - Generate a song")
    print(f"   GET  {tunnel_url}/status/{{job_id}} - Check status")
    print(f"   GET  {tunnel_url}/download/{{job_id}} - Download result")
    print("\n" + "="*60)
    print("⏳ Keep this notebook running to maintain the server")
    print("="*60)
else:
    print("❌ Failed to start tunnel")

## Step 6: Test the API (Optional)

In [ ]:
# Test the API is responding
import requests

if tunnel_url := tunnel_mgr.get_tunnel_url():
    # Health check
    response = requests.get(f"{tunnel_url}/health")
    print(f"Health check: {response.json()}")
    
    # List presets
    response = requests.get(f"{tunnel_url}/presets")
    print(f"\nAvailable presets: {response.json()['presets']}")
    
    # Get API key
    response = requests.get(f"{tunnel_url}/api-key")
    api_key = response.json()['api_key']
    print(f"\nAPI Key: {api_key}")
    print("\n✅ API is working!")
else:
    print("Tunnel not started yet")

## Example: Generate a Song via API

In [ ]:
# Example usage (uncomment to test)
# 
# import requests
# 
# if tunnel_url := tunnel_mgr.get_tunnel_url():
#     payload = {
#         "title": "My AI Song",
#         "lyrics": """
#         [Verse 1]
#         In the digital dreams we weave
#         Through the code we believe
#         
#         [Chorus]
#         Singing loud for all to hear
#         The future of music is here
#         """,
#         "style_preset": "synthwave_retro",
#         "target_duration_minutes": 5.0
#     }
#     
#     response = requests.post(f"{tunnel_url}/generate_song", json=payload)
#     job_data = response.json()
#     print(f"Job created: {job_data}")
#     
#     # Poll for completion
#     job_id = job_data['job_id']
#     while True:
#         status_resp = requests.get(f"{tunnel_url}/status/{job_id}")
#         status = status_resp.json()
#         print(f"Status: {status['status']} - {status.get('progress', {})}")
#         
#         if status['status'] in ['completed', 'failed', 'cancelled']:
#             break
#         
#         time.sleep(10)
#     
#     # Download result
#     if status['status'] == 'completed':
#         download_resp = requests.get(f"{tunnel_url}/download/{job_id}")
#         with open(f"/kaggle/working/{job_id}.wav", 'wb') as f:
#             f.write(download_resp.content)
#         print(f"Song saved to /kaggle/working/{job_id}.wav")